![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 16 -- Lab 1: MNIST in PyTorch

The 60,000 handwritten digits of MNIST, 28 x 28 pixels each. You load them with torchvision, flatten them into rows, build the 784-128-10 network of the deck, and write the training loop as it is really written: mini-batches from a DataLoader, a validation pass every epoch, a checkpoint of the best epoch, and the train-against-validation curves that tell you what happened.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** MNIST (LeCun et al., 1998), downloaded by `torchvision.datasets.MNIST` in the setup cell (about 10 MB): 60,000 training and 10,000 test images, labels 0 to 9. You split the training part into 50,000 for training and 10,000 for validation.

---
# Setup

In [ ]:
# torch, torchvision, matplotlib and scikit-learn are preinstalled on Colab. Uncomment if an import fails.
# !pip install -q torch torchvision matplotlib scikit-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
mnist_train = datasets.MNIST(root="data", train=True, download=True, transform=transforms.ToTensor())
mnist_test = datasets.MNIST(root="data", train=False, download=True, transform=transforms.ToTensor())
img, label = mnist_train[0]
print(len(mnist_train), len(mnist_test), img.shape, img.dtype, label)

---
# Part 1 -- Look, flatten, split

The *Tensor Shapes in Code* and *TensorDataset and DataLoader in Code* slides.

## Task 1: Look at the digits

Show the first 10 training images in a row: `mnist_train[i]` gives `(img, label)`, `img[0]` is the 28 x 28 pixel matrix (`imshow(..., cmap="gray_r")`), the label is the title, axes off.

In [ ]:
# Your code here


## Task 2: Flatten

A network reads rows, not pictures. Take `img` from the setup cell (shape `(1, 28, 28)`) and flatten it with `img.reshape(-1)`: print the new shape (784). Then build a fake batch `batch = torch.rand(64, 1, 28, 28)` and flatten it to one row per image with `batch.reshape(64, -1)`: print its shape. Finally check that `nn.Flatten()(batch).shape` gives the same.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert flat.shape == (784,) and nn.Flatten()(batch).shape == (64, 784), 'flatten: reshape(-1) for one image, reshape(64, -1) or nn.Flatten() for a batch'
print('Task 2 passed')

## Task 3: Split and load

Split `mnist_train` into 50,000 training and 10,000 validation images with `random_split(mnist_train, [50000, 10000], generator=torch.Generator().manual_seed(0))`, naming the parts `train_set, val_set`. Build `train_loader` (batch 64, `shuffle=True`), `val_loader` (batch 1000) and `test_loader` over `mnist_test` (batch 1000). Print `len(train_loader)` (782) and the shapes of the first training batch.

Syntax hint (the shape of the call, not the answer):

```python
part_a, part_b = random_split(dataset, [n_a, n_b], generator=torch.Generator().manual_seed(0))
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(train_set) == 50000 and len(val_set) == 10000 and len(train_loader) == 782, 'split: 50,000 / 10,000, batches of 64 for training'
print('Task 3 passed')

---
# Part 2 -- The network and the untrained loss

The *nn.Linear and nn.Sequential* (Day 15) and *Cross-Entropy, With Numbers* slides.

## Task 4: The network

Build `model = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10)).to(device)`: the flatten layer turns each `(1, 28, 28)` image into 784 numbers, so the loader's images go in as they are. Count the parameters (expected 101,770) and run one batch through it: the output shape must be `(64, 10)`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert n_params == 101770 and model(xb.to(device)).shape == (64, 10), 'model: Flatten, Linear(784, 128), ReLU, Linear(128, 10)'
print('Task 4 passed')

## Task 5: An evaluate function

Write `evaluate(model, loader)` that returns `(mean_loss, accuracy)` over a loader: `model.eval()`, inside `torch.no_grad()` loop over the batches (move them to `device`), sum `loss_fn(logits, yb).item() * len(yb)` and the number of correct `argmax` predictions, and divide both by the number of samples. Use `loss_fn = nn.CrossEntropyLoss()`. Print the untrained model's validation loss and accuracy: expected about 2.30 (ln 10) and about 0.1.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert 2.15 < val_loss < 2.5 and val_acc < 0.25, 'evaluate: mean cross-entropy and accuracy over the loader, in eval mode and no_grad'
print('Task 5 passed')

---
# Part 3 -- The loop, with a checkpoint

The *Loop in Full* and *Validation Inside the Loop: Keep the Best Epoch* slides.

## Task 6: The training loop

Write `fit(model, epochs, lr)`: `opt = torch.optim.SGD(model.parameters(), lr=lr)`; for each epoch, `model.train()`, loop over `train_loader` with the four moves (predict, loss, `zero_grad` + `backward`, `step`), accumulating the mean training loss; then `evaluate` on `val_loader`; append `(train_loss, val_loss, val_acc)` to a list `history`; when `val_acc` beats the best so far, save a checkpoint with `torch.save(model.state_dict(), "best_mnist.pt")` and remember the epoch. Print one line per epoch and return `history`. Run `history = fit(model, 8, 0.1)` (about two minutes on CPU).

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
import os
assert len(history) == 8 and history[-1][2] > 0.95 and os.path.exists('best_mnist.pt'), 'fit: 8 epochs, validation accuracy above 0.95, checkpoint saved'
print('Task 6 passed')

## Task 7: Train against validation

Plot the training loss and the validation loss against the epoch on the left, and the validation accuracy on the right (two subplots, labels, titles, a legend). Is the validation loss still falling at epoch 8? How far apart are the two loss curves?

In [ ]:
# Your code here


---
# Part 4 -- The checkpoint on the test set

## Task 8: Load the best epoch and test it

Create a fresh model with the same layers, load the checkpoint with `model.load_state_dict(torch.load("best_mnist.pt"))`, and evaluate it on `test_loader`. Print the test loss and accuracy as `test_loss, test_acc`: expected accuracy about 0.97, close to the validation number of the best epoch.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert test_acc > 0.95, 'checkpoint: load_state_dict(torch.load("best_mnist.pt")), then evaluate on test_loader'
print('Task 8 passed')

## Task 9: The confusion matrix

Collect the predictions on the whole test set (`argmax` of the logits, inside `torch.no_grad()`) into a NumPy array `pred_test`, and print `confusion_matrix(y_test, pred_test)` where `y_test = mnist_test.targets.numpy()`. Which two digits are confused most often?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert pred_test.shape == (10000,) and (pred_test == y_test).mean() > 0.95, 'predictions: one label per test image, in test order'
print('Task 9 passed')

## Task 10: Look at the mistakes

Find the indices where `pred_test != y_test` and show the first 10 wrong images with the title `pred / true`. What makes them hard?

In [ ]:
# Your code here


## Task 11: The invented columns

Take the first layer's weights, `best_model[1].weight.detach().cpu().numpy()` (shape 128 x 784), and show the first 8 rows as 28 x 28 images (`cmap="RdBu_r"`, axes off): the templates of Day 15, now on full-size digits.

In [ ]:
# Your code here


---
## Task 12: Reflect

Why does the loop save the checkpoint when the *validation* accuracy improves, and not when the training loss falls? The two loss curves stayed close for 8 epochs. Name two changes that would make them separate (Day 16's dashboard), and the one number you would watch to notice it. The flatten layer throws away where each pixel sits. What kind of mistake in Task 10 could that explain, and which week fixes it?

*Your answers here*

---
## Conclusion

- **Images as rows**: `transforms.ToTensor()` gave `(1, 28, 28)` tensors, `nn.Flatten()` turned a batch into a 64 x 784 table, and the 784-128-10 network read it.
- **The loop in full**: a DataLoader of 782 batches, the four moves, `evaluate()` in eval mode and `no_grad`, and a checkpoint saved whenever the validation accuracy improved.
- **The dashboard**: the train and validation curves stayed together, so no overfitting after 8 epochs; the checkpoint scored about the same on the test set as on validation.
- **Reading the model**: the confusion matrix and the wrong images showed where it fails, and the first layer's templates showed what it looks at.

---

Made By **Sattam Altwaim**